# Affinity-Guided LIVECell Training

Clone the repository, install dependencies, train on the attached LIVECell dataset, and copy the run artifacts to Kaggle output.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/EugenTheMachine/some_model.git"
REPOSITORY_DIR = Path("/kaggle/working/some_model_repo")
DATASET_SOURCE = "eugenfromkharkov/livecell-cellseg1-a172"
DATASET_SLUG = DATASET_SOURCE.split("/", 1)[1]
DATASET_ROOT = Path("/kaggle/input") / DATASET_SLUG

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPOSITORY_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPOSITORY_DIR / "src"))

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A Kaggle GPU is required.")
GPU_NAME = torch.cuda.get_device_name(0)
if "T4" not in GPU_NAME.upper():
    raise RuntimeError(f"Expected a Tesla T4 GPU, found {GPU_NAME!r}.")
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset is not mounted at {DATASET_ROOT}.")
print(f"Using {GPU_NAME}; dataset: {DATASET_ROOT}")

In [ ]:
from model import InstanSegModel

epochs = int(os.environ.get("KAGGLE_TRAIN_EPOCHS", "20"))
imgsz = int(os.environ.get("KAGGLE_TRAIN_IMGSZ", "512"))
train_data_ratio = float(os.environ.get("KAGGLE_TRAIN_DATA_RATIO", "1.0"))
if imgsz <= 0 or imgsz % 16:
    raise ValueError("KAGGLE_TRAIN_IMGSZ must be a positive multiple of 16.")

serialized_datasets = list(DATASET_ROOT.rglob("*_dataset.pth"))
if not serialized_datasets:
    serialized_datasets = list(DATASET_ROOT.rglob("*.pth"))
if len(serialized_datasets) > 1:
    raise ValueError(f"Expected one serialized dataset, found {serialized_datasets}")

trainer = InstanSegModel(
    data_dir=str(DATASET_ROOT if serialized_datasets else DATASET_ROOT),
    dataset=str(serialized_datasets[0]) if serialized_datasets else None,
    experiment_name="kaggle_livecell_ag_mlp",
    imgsz=imgsz,
    train_data_ratio=train_data_ratio,
    seed=42,
    dim_in=1,
    epochs=epochs,
    patience=2,
    batch_size=4,
    val_ratio=0.2,
    test_ratio=0.2,
    val_interval=1,
    warmup_epochs=0,
    resume=False,
    amp=True,
    num_workers=0,
    input_channels=1,
    backbone_channels=[16, 32, 64, 128, 256],
    feature_channels=16,
    num_slots=800,
    mlp_channels=[32, 16],
    use_aggregation=True,
)
history = trainer.train()
print(f"Completed {len(history)} epoch(s).")

In [ ]:
OUTPUT_ROOT = Path("/kaggle/working")
RUNS_SOURCE = REPOSITORY_DIR / "runs"
RUNS_OUTPUT = OUTPUT_ROOT / "runs"
if not RUNS_SOURCE.is_dir():
    raise FileNotFoundError(f"Training artifacts were not created: {RUNS_SOURCE}")
if RUNS_OUTPUT.exists():
    shutil.rmtree(RUNS_OUTPUT)
shutil.copytree(RUNS_SOURCE, RUNS_OUTPUT)
print(f"Artifacts copied to {RUNS_OUTPUT}")